# PV056 — Topic 1: Human Activity Recognition

Dataset: UCI HAR (smartphone inertial sensors)  
Goal: classify 6 activities; beat LSTM baseline (F1-macro > 0.80)

**General (every topic):**

- R1a — visualize class label distribution
- R1b — find/visualize outliers or anomalies
- R2a — use HPO (hyperparameter optimization) to find hyperparameters
- R2b — visualize training progress, train to convergence
- R3a — explore successful/failed cases (confusion matrix, explainability)
- R3b — present results in a suitable form

**Topic 1 specific:**
- T1a — data analysis to uncover tough cases (e.g. T-SNE, error analysis, sequence patterns like "does sitting→climbing stairs co-occur?")
- T1b — improve over the given LSTM baseline (~0.80 F1-macro)

## 1. Setup & Data Loading

In [1]:
# !pip install -r requirements.txt

In [2]:
%reset

In [3]:
import pandas as pd
import numpy as np
import matplotlib
import seaborn
import sklearn as sk
import torch
import optuna


/Users/maksim/Documents/Obsidian Vault/MUNI/Courses/Semester 2/PV056 ML and Data Mining/semestral-project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
df_train = pd.read_csv("./data/train.csv")
df_test = pd.read_csv("./data/test.csv")

print(df_train.shape, df_test.shape)

(7352, 563) (2947, 563)


In [5]:
df_train.head()

,tBodyAcc-mean()-X,tBodyAcc-mean()-Y,tBodyAcc-mean()-Z,tBodyAcc-std()-X,tBodyAcc-std()-Y,tBodyAcc-std()-Z,tBodyAcc-mad()-X,tBodyAcc-mad()-Y,tBodyAcc-mad()-Z,tBodyAcc-max()-X,...,fBodyBodyGyroJerkMag-kurtosis(),"angle(tBodyAccMean,gravity)","angle(tBodyAccJerkMean),gravityMean)","angle(tBodyGyroMean,gravityMean)","angle(tBodyGyroJerkMean,gravityMean)","angle(X,gravityMean)","angle(Y,gravityMean)","angle(Z,gravityMean)",subject,Activity
0,0.288585,-0.020294,-0.132905,-0.995279,-0.983111,-0.913526,-0.995112,-0.983185,-0.923527,-0.934724,...,-0.710304,-0.112754,0.030400,-0.464761,-0.018446,-0.841247,0.179941,-0.058627,1,STANDING
1,0.278419,-0.016411,-0.123520,-0.998245,-0.975300,-0.960322,-0.998807,-0.974914,-0.957686,-0.943068,...,-0.861499,0.053477,-0.007435,-0.732626,0.703511,-0.844788,0.180289,-0.054317,1,STANDING
2,0.279653,-0.019467,-0.113462,-0.995380,-0.967187,-0.978944,-0.996520,-0.963668,-0.977469,-0.938692,...,-0.760104,-0.118559,0.177899,0.100699,0.808529,-0.848933,0.180637,-0.049118,1,STANDING
3,0.279174,-0.026201,-0.123283,-0.996091,-0.983403,-0.990675,-0.997099,-0.982750,-0.989302,-0.938692,...,-0.482845,-0.036788,-0.012892,0.640011,-0.485366,-0.848649,0.181935,-0.047663,1,STANDING
4,0.276629,-0.016570,-0.115362,-0.998139,-0.980817,-0.990482,-0.998321,-0.979672,-0.990441,-0.942469,...,-0.699205,0.123320,0.122542,0.693578,-0.615971,-0.847865,0.185151,-0.043892,1,STANDING


In [6]:
df_test.head()

,tBodyAcc-mean()-X,tBodyAcc-mean()-Y,tBodyAcc-mean()-Z,tBodyAcc-std()-X,tBodyAcc-std()-Y,tBodyAcc-std()-Z,tBodyAcc-mad()-X,tBodyAcc-mad()-Y,tBodyAcc-mad()-Z,tBodyAcc-max()-X,...,fBodyBodyGyroJerkMag-kurtosis(),"angle(tBodyAccMean,gravity)","angle(tBodyAccJerkMean),gravityMean)","angle(tBodyGyroMean,gravityMean)","angle(tBodyGyroJerkMean,gravityMean)","angle(X,gravityMean)","angle(Y,gravityMean)","angle(Z,gravityMean)",subject,Activity
0,0.257178,-0.023285,-0.014654,-0.938404,-0.920091,-0.667683,-0.952501,-0.925249,-0.674302,-0.894088,...,-0.705974,0.006462,0.162920,-0.825886,0.271151,-0.720009,0.276801,-0.057978,2,STANDING
1,0.286027,-0.013163,-0.119083,-0.975415,-0.967458,-0.944958,-0.986799,-0.968401,-0.945823,-0.894088,...,-0.594944,-0.083495,0.017500,-0.434375,0.920593,-0.698091,0.281343,-0.083898,2,STANDING
2,0.275485,-0.026050,-0.118152,-0.993819,-0.969926,-0.962748,-0.994403,-0.970735,-0.963483,-0.939260,...,-0.640736,-0.034956,0.202302,0.064103,0.145068,-0.702771,0.280083,-0.079346,2,STANDING
3,0.270298,-0.032614,-0.117520,-0.994743,-0.973268,-0.967091,-0.995274,-0.974471,-0.968897,-0.938610,...,-0.736124,-0.017067,0.154438,0.340134,0.296407,-0.698954,0.284114,-0.077108,2,STANDING
4,0.274833,-0.027848,-0.129527,-0.993852,-0.967445,-0.978295,-0.994111,-0.965953,-0.977346,-0.938610,...,-0.846595,-0.002223,-0.040046,0.736715,-0.118545,-0.692245,0.290722,-0.073857,2,STANDING


## 2. [R1a] Class Label Distribution

In [7]:
df_train['Activity'].value_counts()
# TODO: show histogram here

Activity
LAYING                1407
STANDING              1374
SITTING               1286
WALKING               1226
WALKING_UPSTAIRS      1073
WALKING_DOWNSTAIRS     986
Name: count, dtype: int64

In [8]:
df_test['Activity'].value_counts()
# TODO: show histogram here

Activity
LAYING                537
STANDING              532
WALKING               496
SITTING               491
WALKING_UPSTAIRS      471
WALKING_DOWNSTAIRS    420
Name: count, dtype: int64

In [9]:
df_train['subject'].value_counts()
# TODO: show histogram here

subject
25    409
21    408
26    392
30    383
28    382
27    376
23    372
17    368
16    366
19    360
1     347
29    344
3     341
15    328
6     325
14    323
22    321
11    316
7     308
5     302
8     281
Name: count, dtype: int64

Some subjects are missing in train.

Missing: 2, 4, 9, 10, 12, 13, 18, 20, 24

In [10]:
df_test['subject'].value_counts()
# TODO: show histogram here

subject
24    381
18    364
20    354
13    327
12    320
4     317
2     302
10    294
9     288
Name: count, dtype: int64

Missing subjects are found in test!

### **Notice that subjects in train and test are not the same!**

Test complements the train, be aware of that.

# Build baseline first

In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score

LABEL_COL = 'Activity'
DROP_COLS = ['Activity', 'subject']

X_train = df_train.drop(columns=DROP_COLS).values
y_train = df_train[LABEL_COL].values
X_test  = df_test.drop(columns=DROP_COLS).values
y_test  = df_test[LABEL_COL].values

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)


In [12]:
# Logistic Regression
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_s, y_train)
print(f'LR   F1-macro: {f1_score(y_test, lr.predict(X_test_s), average="macro"):.4f}')

# Random Forest (tree-based, no scaling needed)
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
print(f'RF   F1-macro: {f1_score(y_test, rf.predict(X_test), average="macro"):.4f}')

print('Baseline target: 0.80')


LR   F1-macro: 0.9547
RF   F1-macro: 0.9234
Baseline target: 0.80


In [19]:
from sklearn.metrics import classification_report

print("=== Logistic Regression ===")
print(classification_report(y_test, lr.predict(X_test_s)))

print("=== Random Forest ===")
print(classification_report(y_test, rf.predict(X_test)))


=== Logistic Regression ===
                    precision    recall  f1-score   support

            LAYING       1.00      0.99      0.99       537
           SITTING       0.97      0.88      0.92       491
          STANDING       0.89      0.97      0.93       532
           WALKING       0.94      0.99      0.97       496
WALKING_DOWNSTAIRS       0.99      0.94      0.96       420
  WALKING_UPSTAIRS       0.96      0.95      0.95       471

          accuracy                           0.95      2947
         macro avg       0.96      0.95      0.95      2947
      weighted avg       0.96      0.95      0.95      2947

=== Random Forest ===
                    precision    recall  f1-score   support

            LAYING       1.00      1.00      1.00       537
           SITTING       0.91      0.89      0.90       491
          STANDING       0.90      0.92      0.91       532
           WALKING       0.88      0.97      0.93       496
WALKING_DOWNSTAIRS       0.97      0.86      0

Results are quite surprising. Classical feature engineering + simple classifier outperforms a naive deep learning baseline. Im continuing with performing required steps and would try to improve the baseline

In [13]:
# TODO: to test to shuffle the rows regarding of the subject (mix subjects) and see what changes

In [14]:
# TODO

## 3. [R1b] Outlier / Anomaly Detection

In [15]:
# TODO

## 4. [T1a] Tough Case Analysis

Be aware of differences between subject behaviours within the same class. Visualise per-subject variation within a class.

In [16]:
# TODO: T-SNE, transition/sequence analysis, error analysis

## 5. [R2a/R2b/T1b] Models, HPO, Training

When test different models, be aware of they might require different dataset preparation.
For example XGBoost (doesn't care about scaling (splits are based on rank/thresholds, not distances) vs Neural Networks (care a lot, unscaled features with very different ranges will make training unstable)

**Does the order of windows matter, and how do different models handle (or ignore) that sequence?**
That's the core difference between throwing XGBoost at it vs using an LSTM.

In [17]:
# TODO: baseline LSTM, HPO, improved model, training curves


## 6. [R3a/R3b] Results & Analysis

In [18]:
# TODO: confusion matrix, explainability, results table